In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append("..")

In [3]:
from constrerl.evaluate import (
    eval_submission_6_1_NER,
    eval_submission_6_3_ternary_tag_RE,
    eval_submission_6_4_ternary_mention_RE,
    eval_submission_6_2_binary_tag_RE,
)
from constrerl.erl_schema import convert_to_output, Article
import glob
from pathlib import Path
import json
import pandas as pd
from collections.abc import Callable, Awaitable


In [33]:
report_dir = Path("report_test_ontug")
test_file = Path("./results_test/ONTUG_results.csv")
lbl_xtra = ":test:ontug"
set_mode = True

In [38]:
eval_results: list[dict] = []

import re

model_map = {
    "hermes3b": "Hermes 3B",
    "hermes8b": "Hermes 8B",
    "openai4omni": "Openai 4o mini",
    "openai4": "Openai 4",
}
score_map = {
    "macro_precision": "$P$",
    "macro_recall": "$R$",
    "macro_f1": "$F_1$",
    "micro_precision": "$P_{micro}$",
    "micro_recall": "$R_{micro}$",
    "micro_f1": "$F_{1,micro}$",
}


def test_table_to_df(
    table: pd.DataFrame,
    task: str,
    set_mode: bool = set_mode,
) -> pd.DataFrame:
    eval_results=[]
    for i, row in table.iterrows():
        run_id:str = row["run_id"]
        if row["task_id"] != task:
            continue
        # capitalize the first letter of the name
        model_name = next(
            (v for k, v in model_map.items() if k in run_id.lower()), run_id
        )
        if set_mode:
            model_name ="Hermes 3B + Graphwise"
        result_dict = {
            "Model": model_name,
            "RAG": "\checkmark" if "rag" in run_id else "$\\times$",
            "Reorder": "\checkmark" if "reorder" in run_id else "$\\times$",
            "LoRA": "\checkmark" if "lora" in run_id else "$\\times$",
            # "Low Tokens": "\checkmark"
            # if "low-tokens" in result_file.name
            # else "$\\times$",
            # "Entity Labels": "\checkmark"
            # if "entity-labels" in result_file.name
            # else "$\\times$",
            ** 
            {f"{k}": v for k, v in row.items() if k in score_map},
        }
        if set_mode:
            result_dict["Set"] = "$\cup$" if "union" in run_id else "$\cap$"
        
        # result_dict.update({f"6_2_2_{k}": v for k, v in ternary_tag_score.items()})
        # result_dict.update({f"6_2_3_{k}": v for k, v in ternary_mention_score.items()})
        eval_results.append(result_dict)
    eval_df = pd.DataFrame(eval_results)
    eval_df.rename(score_map, axis=1, inplace=True)
    valid_cols = [
        c
        for c in [
            "Set",
            "Model",
            "RAG",
            "LoRA",
            "LoRA+",
            "Reorder",
            "Low $t$",
            "High $t$",
            "Entities",
            "$k$",
        ]
        if c in eval_df.columns
    ]
    eval_df.set_index(valid_cols, inplace=True)
    eval_df = eval_df.sort_index()
    # if "$F_{1,micro}$" in eval_df.columns:
    #     eval_df = eval_df.sort_values("$F_{1,micro}$")
    return eval_df

result_df=pd.read_csv(test_file, index_col=0)
task_6_2_1_df = test_table_to_df(
    result_df, "T621"
)
task_6_2_2_df = test_table_to_df(
    result_df, "T622"
)
task_6_2_3_df = test_table_to_df(
    result_df, "T623"
)
task_6_2_1_df

<>:39: SyntaxWarning: invalid escape sequence '\c'
<>:40: SyntaxWarning: invalid escape sequence '\c'
<>:41: SyntaxWarning: invalid escape sequence '\c'
<>:52: SyntaxWarning: invalid escape sequence '\c'
<>:52: SyntaxWarning: invalid escape sequence '\c'
<>:39: SyntaxWarning: invalid escape sequence '\c'
<>:40: SyntaxWarning: invalid escape sequence '\c'
<>:41: SyntaxWarning: invalid escape sequence '\c'
<>:52: SyntaxWarning: invalid escape sequence '\c'
<>:52: SyntaxWarning: invalid escape sequence '\c'
/var/folders/w0/5b7f2srd2sb_56zjv17vqnym0000gn/T/ipykernel_77798/1652987304.py:39: SyntaxWarning: invalid escape sequence '\c'
  "RAG": "\checkmark" if "rag" in run_id else "$\\times$",
/var/folders/w0/5b7f2srd2sb_56zjv17vqnym0000gn/T/ipykernel_77798/1652987304.py:40: SyntaxWarning: invalid escape sequence '\c'
  "Reorder": "\checkmark" if "reorder" in run_id else "$\\times$",
/var/folders/w0/5b7f2srd2sb_56zjv17vqnym0000gn/T/ipykernel_77798/1652987304.py:41: SyntaxWarning: invalid esca

,,,,,$P$,$R$,$F_1$,$P_{micro}$,$R_{micro}$,"$F_{1,micro}$"
Set,Model,RAG,LoRA,Reorder,,,,,,
$\cap$,Hermes 3B + Graphwise,$\times$,$\times$,$\times$,0.172093,0.055800,0.079935,0.885714,0.134199,0.233083
$\cup$,Hermes 3B + Graphwise,$\times$,$\times$,$\times$,0.418467,0.407276,0.405673,0.712121,0.610390,0.657343


In [39]:
task_6_2_1_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(precision=2).to_latex(
    report_dir / "task_6_2_1.tex",
    # float_format="%.2f",
    caption="Dev Set Result for Task 6.2.1 for various models and approaches.",
    label=f"tab:task:6_2_1{lbl_xtra}",
    clines="all;data",
    hrules=True
)
task_6_2_1_df

,,,,,$P$,$R$,$F_1$,$P_{micro}$,$R_{micro}$,"$F_{1,micro}$"
Set,Model,RAG,LoRA,Reorder,,,,,,
$\cap$,Hermes 3B + Graphwise,$\times$,$\times$,$\times$,0.172093,0.055800,0.079935,0.885714,0.134199,0.233083
$\cup$,Hermes 3B + Graphwise,$\times$,$\times$,$\times$,0.418467,0.407276,0.405673,0.712121,0.610390,0.657343


In [40]:
task_6_2_2_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(precision=2).to_latex(
    report_dir / "task_6_2_2.tex",
    # float_format="%.2f",
    caption="Further Dev Set Result for Task 6.2.2 for various models and approaches.",
    label=f"tab:task:6_2_2{lbl_xtra}",
    clines="all;data",
    hrules=True
)
task_6_2_2_df

,,,,,$P$,$R$,$F_1$,$P_{micro}$,$R_{micro}$,"$F_{1,micro}$"
Set,Model,RAG,LoRA,Reorder,,,,,,
$\cup$,Hermes 3B + Graphwise,$\times$,$\times$,$\times$,0.425413,0.402453,0.405831,0.705882,0.592593,0.644295


In [41]:
task_6_2_3_df.style.highlight_max(axis=0, props="textbf:--rwrap;").format(precision=2).to_latex(
    report_dir / "task_6_2_3.tex",
    # float_format="%.2f",
    caption="Dev Set Result for Task 6.2.3 for various models and approaches.",
    label=f"tab:task:6_2_3{lbl_xtra}",
    clines="all;data",
    hrules=True
)
task_6_2_3_df

,,,,,$P$,$R$,$F_1$,$P_{micro}$,$R_{micro}$,"$F_{1,micro}$"
Set,Model,RAG,LoRA,Reorder,,,,,,
$\cup$,Hermes 3B + Graphwise,$\times$,$\times$,$\times$,0.258872,0.229349,0.22663,0.352855,0.323056,0.337299
